<a href="https://colab.research.google.com/github/PradeepTiwari12/Seasonal-Agriculture-Performance-Analysis/blob/main/Seasonal_Agriculture_Performance_Analysis_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
"""
=============================================================================
VOIS AICTE Major Project: Seasonal Agriculture Performance Analysis
Batch: 2026-2027 | Course: Data Visualization
File: seasonal_agriculture_analysis.py
Description:
    End-to-end data processing, exploratory data analysis (EDA), statistical
    evaluation (ANOVA, correlation), and publication-quality visualizations
    for seasonal agricultural performance.
=============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ---------------------------------------------------------------------------
# 1. Visualization & Plotting Setup
# ---------------------------------------------------------------------------
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.titlesize": 16,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.sans-serif": "DejaVu Sans",
    "figure.autolayout": True,
    "savefig.dpi": 300
})

OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)



In [34]:

# ---------------------------------------------------------------------------
# 2. Data Loading & Inspection
# ---------------------------------------------------------------------------
def load_and_inspect_data(filepath: str) -> pd.DataFrame:
    """Loads agricultural dataset and inspects basic schema, nulls, and types."""
    print("=" * 80)
    print(f"[+] Loading Dataset from: {filepath}")
    print("=" * 80)

    if not os.path.exists(filepath):
        print(f"[!] Warning: File '{filepath}' not found on local path.")
        print("[*] Generating representative synthetic dataset for full pipeline demonstration...")
        df = generate_synthetic_agricultural_data()
    else:
        df = pd.read_csv(filepath)

    print("\n[+] Data Shape:", df.shape)
    print("\n[+] Data Columns:\n", df.columns.tolist())
    print("\n[+] Data Head (First 5 records):\n", df.head())
    print("\n[+] Data Info:")
    df.info()
    print("\n[+] Missing Values Count:\n", df.isnull().sum())

    return df


def generate_synthetic_agricultural_data(n_samples: int = 1200) -> pd.DataFrame:
    """Fallback generator reproducing the exact schema of seasonal agriculture data."""
    np.random.seed(42)
    seasons = np.random.choice(["Kharif", "Rabi", "Zaid"], size=n_samples, p=[0.48, 0.38, 0.14])
    regions = np.random.choice(["Northern Plains", "Southern Plateau", "Western Coastal", "Eastern Gangetic"], size=n_samples)
    crops_kharif = ["Rice", "Maize", "Cotton", "Soybean", "Groundnut"]
    crops_rabi = ["Wheat", "Barley", "Mustard", "Gram", "Peas"]
    crops_zaid = ["Watermelon", "Cucumber", "Bitter Gourd", "Moong"]

    crops = []
    for s in seasons:
        if s == "Kharif":
            crops.append(np.random.choice(crops_kharif))
        elif s == "Rabi":
            crops.append(np.random.choice(crops_rabi))
        else:
            crops.append(np.random.choice(crops_zaid))

    rainfall = []
    temp = []
    water_usage = []
    fert_usage = []
    crop_yield = []
    cost = []
    revenue = []

    for s in seasons:
        if s == "Kharif":
            rain = np.random.normal(850, 120)
            t = np.random.normal(31, 3.2)
            water = np.random.normal(4800, 500)
            fert = np.random.normal(160, 25)
            y = np.random.normal(3850, 420)
            c = np.random.normal(24500, 2200)
            r = c + (y * np.random.uniform(7.5, 9.8))
        elif s == "Rabi":
            rain = np.random.normal(180, 45)
            t = np.random.normal(19, 2.8)
            water = np.random.normal(3400, 400)
            fert = np.random.normal(140, 20)
            y = np.random.normal(4350, 380)
            c = np.random.normal(21000, 1900)
            r = c + (y * np.random.uniform(8.2, 10.5))
        else:  # Zaid
            rain = np.random.normal(90, 30)
            t = np.random.normal(36, 3.5)
            water = np.random.normal(2800, 350)
            fert = np.random.normal(110, 18)
            y = np.random.normal(2950, 320)
            c = np.random.normal(16800, 1500)
            r = c + (y * np.random.uniform(9.0, 11.2))

        rainfall.append(max(20, rain))
        temp.append(t)
        water_usage.append(water)
        fert_usage.append(fert)
        crop_yield.append(y)
        cost.append(c)
        revenue.append(r)

    df = pd.DataFrame({
        "Farm_ID": [f"FARM_{1000+i}" for i in range(n_samples)],
        "Region": regions,
        "Season": seasons,
        "Crop": crops,
        "Rainfall_mm": np.round(rainfall, 2),
        "Temperature_C": np.round(temp, 2),
        "Water_Usage_kL_per_ha": np.round(water_usage, 2),
        "Fertilizer_Usage_kg_per_ha": np.round(fert_usage, 2),
        "Crop_Yield_kg_per_ha": np.round(crop_yield, 2),
        "Cost_per_ha_INR": np.round(cost, 2),
        "Revenue_per_ha_INR": np.round(revenue, 2),
    })
    df["Profit_per_ha_INR"] = np.round(df["Revenue_per_ha_INR"] - df["Cost_per_ha_INR"], 2)
    df["Profit_Margin_pct"] = np.round((df["Profit_per_ha_INR"] / df["Revenue_per_ha_INR"]) * 100, 2)
    return df



In [35]:

# ---------------------------------------------------------------------------
# 3. Data Cleaning & Preprocessing
# ---------------------------------------------------------------------------
def clean_and_prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    """Handles missing values, standardizes column formats, and computes derived metrics."""
    print("\n" + "=" * 80)
    print("[+] Cleaning and Preprocessing Data")
    print("=" * 80)

    df_clean = df.copy()

    # Handle numeric missing values with median
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if df_clean[col].isnull().sum() > 0:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            print(f"[*] Filled {col} missing values with median: {median_val:.2f}")

    # Handle categorical nulls
    cat_cols = df_clean.select_dtypes(include=["object"]).columns
    for col in cat_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"[*] Filled {col} missing values with mode: {mode_val}")

    # Calculate key economic metrics if absent
    if "Profit_per_ha_INR" not in df_clean.columns and "Revenue_per_ha_INR" in df_clean.columns:
        df_clean["Profit_per_ha_INR"] = df_clean["Revenue_per_ha_INR"] - df_clean["Cost_per_ha_INR"]

    if "Profit_Margin_pct" not in df_clean.columns and "Revenue_per_ha_INR" in df_clean.columns:
        df_clean["Profit_Margin_pct"] = (df_clean["Profit_per_ha_INR"] / df_clean["Revenue_per_ha_INR"]) * 100

    print("[+] Data preprocessing complete. Cleaned shape:", df_clean.shape)
    return df_clean



In [36]:

# ---------------------------------------------------------------------------
# 4. Seasonal Aggregation & Exploratory Analysis
# ---------------------------------------------------------------------------
def compute_seasonal_statistics(df: pd.DataFrame) -> pd.DataFrame:
    """Computes comprehensive mean, std, and aggregate metrics grouped by Season."""
    print("\n" + "=" * 80)
    print("[+] Seasonal Performance Aggregates")
    print("=" * 80)

    metrics = [
        "Crop_Yield_kg_per_ha",
        "Water_Usage_kL_per_ha",
        "Fertilizer_Usage_kg_per_ha",
        "Cost_per_ha_INR",
        "Revenue_per_ha_INR",
        "Profit_per_ha_INR",
        "Profit_Margin_pct"
    ]

    avail_metrics = [m for m in metrics if m in df.columns]
    summary_table = df.groupby("Season")[avail_metrics].agg(["mean", "std", "min", "max"]).round(2)
    print(summary_table)

    # Export summary table to CSV
    summary_table.to_csv(os.path.join(OUTPUT_DIR, "seasonal_summary_metrics.csv"))
    print(f"[+] Saved summary table to '{OUTPUT_DIR}/seasonal_summary_metrics.csv'")

    return summary_table


In [37]:


# ---------------------------------------------------------------------------
# 5. Statistical Hypothesis Testing (ANOVA & Correlation)
# ---------------------------------------------------------------------------
def run_statistical_tests(df: pd.DataFrame):
    """Conducts One-way ANOVA across seasons and computes feature correlations."""
    print("\n" + "=" * 80)
    print("[+] Statistical Significance Testing (One-Way ANOVA across Seasons)")
    print("=" * 80)

    seasons = df["Season"].unique()
    target_vars = ["Crop_Yield_kg_per_ha", "Water_Usage_kL_per_ha", "Profit_per_ha_INR"]

    for var in target_vars:
        if var in df.columns:
            groups = [df[df["Season"] == s][var].dropna() for s in seasons]
            f_stat, p_val = stats.f_oneway(*groups)
            print(f"[-] ANOVA for {var:28s}: F-Stat = {f_stat:8.3f}, p-value = {p_val:.4e}")
            if p_val < 0.05:
                print(f"    --> Statistically significant variation across seasons (p < 0.05).")
            else:
                print(f"    --> No statistically significant variation across seasons (p >= 0.05).")



In [38]:

# ---------------------------------------------------------------------------
# 6. Data Visualizations Generation
# ---------------------------------------------------------------------------
def generate_visualizations(df: pd.DataFrame):
    """Generates charts answering core research questions."""
    print("\n" + "=" * 80)
    print("[+] Generating Visualizations for Presentation Slides...")
    print("=" * 80)

    palette = {"Kharif": "#2d6a4f", "Rabi": "#1b4332", "Zaid": "#d4a373"}

    # Figure 1: Crop Yield by Season (Bar / Boxplot)
    plt.figure(figsize=(9, 5.5))
    ax = sns.boxplot(x="Season", y="Crop_Yield_kg_per_ha", data=df, palette=palette, boxprops=dict(alpha=0.85))
    sns.stripplot(x="Season", y="Crop_Yield_kg_per_ha", data=df, color="black", alpha=0.15, jitter=0.2, size=4)
    plt.title("Seasonal Variation in Crop Yield (kg/ha)", fontweight="bold", pad=15)
    plt.xlabel("Agricultural Season", fontweight="bold")
    plt.ylabel("Crop Yield (kg/ha)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig1_crop_yield_by_season.png"))
    plt.close()
    print("[+] Generated: fig1_crop_yield_by_season.png")

    # Figure 2: Resource Utilization (Water & Fertilizer Usage)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    sns.barplot(x="Season", y="Water_Usage_kL_per_ha", data=df, ax=ax1, palette=palette, ci=None, edgecolor="black")
    ax1.set_title("Mean Water Usage Across Seasons (kL/ha)", fontweight="bold")
    ax1.set_ylabel("Water Usage (kL/ha)")

    sns.barplot(x="Season", y="Fertilizer_Usage_kg_per_ha", data=df, ax=ax2, palette=palette, ci=None, edgecolor="black")
    ax2.set_title("Mean Fertilizer Usage Across Seasons (kg/ha)", fontweight="bold")
    ax2.set_ylabel("Fertilizer (kg/ha)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig2_resource_utilization.png"))
    plt.close()
    print("[+] Generated: fig2_resource_utilization.png")

    # Figure 3: Economic Profitability (Cost vs Revenue vs Net Profit)
    econ_summary = df.groupby("Season")[["Cost_per_ha_INR", "Revenue_per_ha_INR", "Profit_per_ha_INR"]].mean().reset_index()
    econ_melted = pd.melt(econ_summary, id_vars=["Season"], var_name="Metric", value_name="Amount_INR")

    plt.figure(figsize=(10, 5.5))
    sns.barplot(x="Season", y="Amount_INR", hue="Metric", data=econ_melted, palette="Set2", edgecolor="black")
    plt.title("Economic Performance Breakdown by Season (INR per ha)", fontweight="bold", pad=15)
    plt.xlabel("Season", fontweight="bold")
    plt.ylabel("Amount (INR)", fontweight="bold")
    plt.legend(title="Financial Indicator")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig3_economic_performance.png"))
    plt.close()
    print("[+] Generated: fig3_economic_performance.png")

    # Figure 4: Correlation Matrix
    num_df = df.select_dtypes(include=[np.number]).drop(columns=["Farm_ID"], errors="ignore")
    plt.figure(figsize=(9, 7))
    sns.heatmap(num_df.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, cbar_kws={"label": "Pearson Correlation"})
    plt.title("Feature Correlation Matrix - Agriculture Performance", fontweight="bold", pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig4_correlation_heatmap.png"))
    plt.close()
    print("[+] Generated: fig4_correlation_heatmap.png")



In [39]:

# ---------------------------------------------------------------------------
# 7. Main Execution Pipeline
# ---------------------------------------------------------------------------
def main():
    print("=" * 80)
    print("VOIS AICTE Major Project - Seasonal Agriculture Performance Analysis")
    print("=" * 80)

    # Search for user-provided CSV or fallback
    target_files = [

        "seasonal_agriculture_performance_dataset.csv",

    ]

    chosen_file = None
    for f in target_files:
        if os.path.exists(f):
            chosen_file = f
            break

    if chosen_file is None:
        chosen_file = "seasonal_agriculture_performance_dataset.csv"

    # Execute full workflow
    df_raw = load_and_inspect_data(chosen_file)
    df_clean = clean_and_prepare_data(df_raw)
    compute_seasonal_statistics(df_clean)
    run_statistical_tests(df_clean)
    generate_visualizations(df_clean)

    print("\n" + "=" * 80)
    print("[✓] Analysis completed successfully! Results and charts generated in:", OUTPUT_DIR)
    print("=" * 80)


if __name__ == "__main__":
    main()

VOIS AICTE Major Project - Seasonal Agriculture Performance Analysis
[+] Loading Dataset from: seasonal_agriculture_performance_dataset.csv
[!] Warning: File 'seasonal_agriculture_performance_dataset.csv' not found on local path.
[*] Generating representative synthetic dataset for full pipeline demonstration...

[+] Data Shape: (1200, 13)

[+] Data Columns:
 ['Farm_ID', 'Region', 'Season', 'Crop', 'Rainfall_mm', 'Temperature_C', 'Water_Usage_kL_per_ha', 'Fertilizer_Usage_kg_per_ha', 'Crop_Yield_kg_per_ha', 'Cost_per_ha_INR', 'Revenue_per_ha_INR', 'Profit_per_ha_INR', 'Profit_Margin_pct']

[+] Data Head (First 5 records):
      Farm_ID            Region  Season       Crop  Rainfall_mm  Temperature_C  \
0  FARM_1000   Northern Plains  Kharif  Groundnut       930.74          32.89   
1  FARM_1001  Eastern Gangetic    Zaid      Moong       119.66          31.27   
2  FARM_1002   Western Coastal    Rabi       Peas       175.35          14.40   
3  FARM_1003   Northern Plains    Rabi     Bar

/tmp/ipykernel_914/515410863.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x="Season", y="Crop_Yield_kg_per_ha", data=df, palette=palette, boxprops=dict(alpha=0.85))


[+] Generated: fig1_crop_yield_by_season.png


/tmp/ipykernel_914/515410863.py:26: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x="Season", y="Water_Usage_kL_per_ha", data=df, ax=ax1, palette=palette, ci=None, edgecolor="black")
/tmp/ipykernel_914/515410863.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="Season", y="Water_Usage_kL_per_ha", data=df, ax=ax1, palette=palette, ci=None, edgecolor="black")
/tmp/ipykernel_914/515410863.py:30: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x="Season", y="Fertilizer_Usage_kg_per_ha", data=df, ax=ax2, palette=palette, ci=None, edgecolor="black")
/tmp/ipykernel_914/515410863.py:30: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue`

[+] Generated: fig2_resource_utilization.png
[+] Generated: fig3_economic_performance.png
[+] Generated: fig4_correlation_heatmap.png

[✓] Analysis completed successfully! Results and charts generated in: analysis_outputs
